# EDA — Deep Past Initiative: Akkadian → English

Изучаем корпус соревнования и выводим три ключевых факта, на которых строится весь пайплайн:

1. **Тестовые транслитерации испорчены детерминированным посимвольным шифром** (артефакт шрифта/OCR публикаций): `š→a`, `ṭ→m`, `ḫ→+`, индексы знаков `4→„`, `5→…`, скобки `{}→()`.
2. **Орфография различается между файлами**: `train.csv` использует юникодные подстрочные индексы (`bi₄`), `published_texts.csv` — ASCII-цифры (`bi4`). Тест следует орфографии published + шифр.
3. **`Sentences_Oare_FirstWord_LinNum.csv` даёт ~8k дополнительных пар на уровне предложений** — их можно заякорить в транслитерациях монотонным поиском первого слова (85% текстов выравниваются полностью).

In [ ]:
import csv, difflib, re, statistics
from itertools import groupby

csv.field_size_limit(10**9)
DATA = "../data/"

def rows(f):
    with open(DATA + f, encoding="utf-8-sig") as fh:
        return list(csv.DictReader(fh))

train = rows("train.csv")
test = rows("test.csv")
published = rows("published_texts.csv")
sentences = rows("Sentences_Oare_FirstWord_LinNum.csv")
print(f"train: {len(train)} docs | test (visible): {len(test)} rows | "
      f"published_texts: {len(published)} | sentences: {len(sentences)}")

## 1. Шифр в тестовых транслитерациях

Видимые 4 тестовых строки — это фрагменты (диапазоны строк) одной таблички. Ищем её чистую версию в `published_texts.csv` по неиспорченным n-граммам (`wa-bar-ra-tim`, `KÙ.AN`) и выравниваем посимвольно.

In [ ]:
hits = [r for r in published
        if "wa-bar-ra-tim" in r["transliteration"] and "KÙ.AN" in r["transliteration"]]
assert len(hits) == 1
clean = hits[0]["transliteration"]
corrupt = " ".join(r["transliteration"] for r in test)
print("clean tablet:", hits[0]["label"], "| len clean/corrupt:", len(clean), len(corrupt))

sm = difflib.SequenceMatcher(None, clean, corrupt, autojunk=False)
subs = {}
for tag, i1, i2, j1, j2 in sm.get_opcodes():
    if tag != "equal":
        subs[(clean[i1:i2], corrupt[j1:j2])] = subs.get((clean[i1:i2], corrupt[j1:j2]), 0) + 1
for (c, k), n in sorted(subs.items(), key=lambda x: -x[1]):
    print(f"  {c!r} -> {k!r}  x{n}")

Результат (длины совпадают 678=678 → это чистая посимвольная замена):

```
'š' -> 'a'  x25
'4' -> '„'  x6
'ṭ' -> 'm'  x5
'{' -> '('  x2
'}' -> ')'  x2
'5' -> '…'  x1
'ḫ' -> '+'  x1
```

Замены **необратимы** (`a` в тесте — это и настоящая `a`, и бывшая `š`), поэтому стратегия — не чистить тест, а **портить train-источник так же** и учить модель на испорченном входе.

## 2. Различия орфографии train.csv vs published_texts.csv

In [ ]:
tr_chars = set("".join(r["transliteration"] for r in train))
pt_chars = set("".join(r["transliteration"] for r in published if r["transliteration"]))
te_chars = set("".join(r["transliteration"] for r in test))
print("подстрочные индексы в train:", sorted(c for c in tr_chars if "₀" <= c <= "₉"))
print("подстрочные индексы в published:", sorted(c for c in pt_chars if "₀" <= c <= "₉"))
print("символы теста, отсутствующие в train:", sorted(te_chars - tr_chars))

`train.csv` пишет `bi₄`/`il₅` (юникод), `published_texts.csv` — `bi4`/`il5` (ASCII). Тест = ASCII-вариант + шифр. Значит, нормализатор сначала приводит всё к единой орфографии (ASCII-индексы), затем (для аугментации) применяет шифр.

## 3. Выравнивание предложений (доп. параллельные данные)

`first_word_obj_in_text` считает объекты БД (включая номера строк), а не слова — напрямую не используется. Вместо этого: сортируем предложения текста по `sentence_obj_in_text` и монотонно ищем `first_word_spelling` в списке слов транслитерации.

In [ ]:
SUB = str.maketrans("₀₁₂₃₄₅₆₇₈₉", "0123456789")

def norm_word(w):
    w = w.translate(SUB).lower()
    w = re.sub(r"[<>\[\]⸢⸣!?#*…]", "", w)
    return re.sub(r"\d+$", "", w)

tr_map = {r["oare_id"]: r["transliteration"] for r in train}
pt_map = {r["oare_id"]: r["transliteration"] for r in published if r["transliteration"]}
sentences.sort(key=lambda r: (r["text_uuid"], int(r["sentence_obj_in_text"] or 0)))

n_texts = n_full = n_sent = n_anchored = 0
sent_lens = []
for tid, grp in groupby(sentences, key=lambda r: r["text_uuid"]):
    grp = list(grp)
    text = tr_map.get(tid) or pt_map.get(tid)
    if text is None:
        continue
    words = text.split()
    nwords = [norm_word(w) for w in words]
    pos, anchors, ok = 0, [], 0
    for s in grp:
        sp = norm_word(s["first_word_spelling"]) if s["first_word_spelling"] else ""
        found = next((j for j in range(pos, len(nwords)) if sp and nwords[j] == sp), -1)
        anchors.append(found)
        if found >= 0:
            ok, pos = ok + 1, found + 1
    n_texts += 1; n_sent += len(grp); n_anchored += ok
    if ok == len(grp):
        n_full += 1
        for k, a in enumerate(anchors):
            end = anchors[k + 1] if k + 1 < len(anchors) else len(words)
            sent_lens.append(len(" ".join(words[a:end])))

print(f"текстов полностью выравнено: {n_full}/{n_texts} ({100*n_full/n_texts:.1f}%)")
print(f"предложений заякорено: {n_anchored}/{n_sent} ({100*n_anchored/n_sent:.1f}%)")
print(f"длина источника предложения: медиана {statistics.median(sent_lens)}, "
      f"p90 {sorted(sent_lens)[int(0.9*len(sent_lens))]} символов")

Итог: **1210/1417 текстов (85.4%)** выравниваются полностью, **8080/8476 предложений (95.3%)**. Из них 1164 текста отсутствуют в `train.csv` → тысячи новых пар.

Медианная длина предложения ~63 символа, а тестовые фрагменты — 150–300 символов (диапазоны из 4–10 строк таблички). Поэтому в `data_prep.py` соседние предложения группируются в чанки по 1–5 штук, имитируя распределение длин теста.

## Выводы для пайплайна

| Факт | Решение в пайплайне |
|---|---|
| Тест испорчен шифром `š→a, ṭ→m, ḫ→+, 4→„, 5→…, {}→()` | Аугментация: портим источник train так же (`normalize.py`) |
| Орфографии train/published различаются | Единая нормализация: NFC, ASCII-индексы, унификация скобок |
| 8k предложений выравниваются по первому слову | `data_prep.py` строит чанк-пары из предложений + 1561 док-пара |
| Тест — диапазоны строк, не целые доки | Чанки из 1–5 предложений имитируют длину теста |